# Projeto 2 — Aprendizado de Máquina: Olist
**Previsão de Tempo de Entrega de Pedidos (Regressão)**

Autores: Kaike Brito Leitão · Enrico Santos Navajas · Mario · Pedro Chaves · Ian Santos · Luiz Chaves
Disciplina: T326 — Ciência dos Dados · Prof. Caio Ponte · Turma 16/17

In [ ]:
# =============================================================================
# IMPORTS E CONFIGURAÇÃO GLOBAL
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, logging
from pathlib import Path
from typing import Dict, List
from IPython.display import display

from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger(__name__)

RAW          = Path("../dataframes/raw/")       # CSVs brutos do Kaggle
OUT          = Path("../dataframes/processed/")  # figuras e CSVs processados
OUT.mkdir(exist_ok=True)

RANDOM_STATE = 42    # semente fixa para reprodutibilidade
TEST_SIZE    = 0.20  # 20% para avaliação final (~19.115 pedidos)
OUTLIER_P99  = 46    # P99 do target: acima de 46 dias = outlier extremo
PALETA       = "#2563EB"

plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,   # remove eixo superior (visual mais limpo)
    "axes.spines.right": False, # remove eixo direito
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
print("Setup concluido")

---
## Secao 1 - Carregamento das Tabelas

In [ ]:
# =============================================================================
# SECAO 1 - CARREGAMENTO DAS 8 TABELAS
# =============================================================================
# Hub central: orders. Todas as demais se unem por order_id ou chaves secundarias.
def carregar_tabelas(path: Path) -> Dict[str, pd.DataFrame]:
    arquivos = {
        "orders":      "olist_orders_dataset.csv",      # hub: timestamps e status
        "order_items": "olist_order_items_dataset.csv",  # itens: preco, frete, seller
        "payments":    "olist_order_payments_dataset.csv",# pagamentos por pedido
        "products":    "olist_products_dataset.csv",     # dimensoes e categoria do produto
        "sellers":     "olist_sellers_dataset.csv",      # localizacao do vendedor
        "customers":   "olist_customers_dataset.csv",    # localizacao do cliente
        "geolocation": "olist_geolocation_dataset.csv",  # lat/lng por CEP
        "translation": "product_category_name_translation.csv", # pt->en categorias
    }
    dfs = {}
    for nome, arq in arquivos.items():
        dfs[nome] = pd.read_csv(path / arq)
        logger.info(f"{nome}: {dfs[nome].shape}")
    return dfs

dfs = carregar_tabelas(RAW)
display(pd.DataFrame([{"Tabela": k, "Linhas": v.shape[0], "Colunas": v.shape[1]} for k, v in dfs.items()]))

---
## Secao 2 - Engenharia de Features (38 features em 6 dominios)

In [ ]:
# =============================================================================
# MAPEAMENTO UF -> MACRORREGIAO
# =============================================================================
# Usado para criar regiao_cliente e regiao_vendedor.
# Macrorregioes capturam padroes logisticos que UFs isoladas nao revelam:
# Norte tem infraestrutura precaria; Sudeste concentra a maioria dos vendedores.
REGIAO_MAP: Dict[str, str] = {
    "AC":"Norte","AM":"Norte","AP":"Norte","PA":"Norte","RO":"Norte","RR":"Norte","TO":"Norte",
    "AL":"Nordeste","BA":"Nordeste","CE":"Nordeste","MA":"Nordeste","PB":"Nordeste",
    "PE":"Nordeste","PI":"Nordeste","RN":"Nordeste","SE":"Nordeste",
    "DF":"Centro-Oeste","GO":"Centro-Oeste","MS":"Centro-Oeste","MT":"Centro-Oeste",
    "ES":"Sudeste","MG":"Sudeste","RJ":"Sudeste","SP":"Sudeste",
    "PR":"Sul","RS":"Sul","SC":"Sul",
}

def haversine_vec(lat1: np.ndarray, lon1: np.ndarray,
                  lat2: np.ndarray, lon2: np.ndarray) -> np.ndarray:
    """
    Distancia geodesica em km entre dois pontos geograficos (vetorizada).

    POR QUE HAVERSINE E NAO EUCLIDIANA?
    A distancia euclidiana em graus (sqrt(Dlat^2+Dlng^2)) ignora que a Terra e
    esferica. Um grau de longitude vale distancias diferentes dependendo da latitude:
      - Equador (0 graus): 1 grau lng = 111 km
      - Sul do Brasil (30 graus S): 1 grau lng = 96 km
    Em rotas longas como SP->AM, o erro euclidiano chega a 15%.

    FORMULA DE HAVERSINE:
    1) Converter graus -> radianos (funcoes trig exigem radianos)
    2) Calcular angulo central esferico:
       a = sin^2(Dlat/2) + cos(lat1)*cos(lat2)*sin^2(Dlng/2)
    3) Converter angulo -> distancia:
       d = 2*R*arcsin(sqrt(a))   onde R = 6371 km

    np.clip(a, 0, 1) evita NaN por erros numericos de ponto flutuante.
    """
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def construir_dataset(dfs: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    """
    Integra as 8 tabelas Olist em 1 linha por pedido entregue com 38 features.
    Shape esperado (antes da limpeza): ~96.000 linhas x 39 colunas.
    """

    # ── [1] ORDERS: filtrar entregues e converter timestamps ──────────────────
    orders = dfs["orders"].copy()
    for col in ["order_purchase_timestamp", "order_delivered_customer_date",
                "order_estimated_delivery_date", "order_approved_at",
                "order_delivered_carrier_date"]:
        orders[col] = pd.to_datetime(orders[col])  # string -> datetime para calculo de diferenca

    # Manter apenas status="delivered" com data de entrega registrada
    df = orders[orders["order_status"] == "delivered"].dropna(
        subset=["order_delivered_customer_date"]).copy()

    # ── [2] TARGET: dias_entrega ───────────────────────────────────────────────
    # .dt.days extrai o campo de dias do timedelta resultante da subtracao de datas
    # Exemplo: 2018-02-01 - 2018-01-20 = timedelta(12) -> dias_entrega = 12
    df["dias_entrega"] = (
        df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
    ).dt.days

    # ── [3] FEATURES TEMPORAIS ────────────────────────────────────────────────

    # estimativa_prazo | r = +0.43 (2a feature mais correlacionada)
    # Prazo prometido pela Olist. Alta correlacao porque a Olist usa dados
    # logisticos internos para estimar o prazo - informacao que o modelo aproveita.
    df["estimativa_prazo"] = (
        df["order_estimated_delivery_date"] - df["order_purchase_timestamp"]
    ).dt.days

    # dia_semana_compra | 0=Seg, 6=Dom
    # Compras no fim de semana tem aprovacao de pagamento atrasada (bancos fechados)
    df["dia_semana_compra"] = df["order_purchase_timestamp"].dt.dayofweek

    # hora_compra | 0-23h
    # Compras tarde da noite sao processadas no proximo dia util
    df["hora_compra"] = df["order_purchase_timestamp"].dt.hour

    # mes_compra | 1=Jan, 12=Dez
    # Captura sazonalidade logistica: Black Friday (Nov), Natal (Dez) sobrecarregam a rede
    # EDA mostra: Jun-Ago mais rapido; Jan-Mar mais lento
    df["mes_compra"] = df["order_purchase_timestamp"].dt.month

    # dias_ate_aprova_h | r = +0.10
    # Horas entre compra e aprovacao do pagamento.
    # .total_seconds() / 3600 converte timedelta para horas decimais
    # (ex: 2h30min = 150 segundos / 3600 = 2.5h)
    # Aprovacao lenta (especialmente boleto: D+1 a D+3 util) atrasa o inicio
    # da preparacao do pedido pelo vendedor.
    df["dias_ate_aprova_h"] = (
        df["order_approved_at"] - df["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600

    # fim_de_semana | 0 ou 1
    # 1 = sabado (5) ou domingo (6), 0 = dia util
    # Bancos nao processam pagamentos no fim de semana -> pedidos entram na fila
    # de preparacao apenas na segunda-feira
    df["fim_de_semana"] = df["dia_semana_compra"].isin([5, 6]).astype(int)

    # periodo_dia | 0=madrugada, 1=manha, 2=tarde, 3=noite
    # pd.cut divide hora_compra em 4 intervalos com os bins especificados
    # right=False: inclui o limite esquerdo, exclui o direito [a, b)
    # Compras de madrugada/noite chegam ao vendedor apenas no proximo periodo util
    df["periodo_dia"] = pd.cut(
        df["hora_compra"], bins=[0, 6, 12, 18, 24],
        labels=[0, 1, 2, 3], right=False
    ).astype(float)  # float: OrdinalEncoder nao aceita Categorical

    df = df[["order_id", "customer_id", "dias_entrega", "estimativa_prazo",
             "dia_semana_compra", "hora_compra", "mes_compra", "dias_ate_aprova_h",
             "fim_de_semana", "periodo_dia"]]

    # ── [4] ORDER ITEMS -> 1 linha por pedido ─────────────────────────────────
    # order_items tem 1 linha POR ITEM -> agregar por order_id
    items = dfs["order_items"]
    ia = items.groupby("order_id").agg(
        n_items         = ("order_item_id", "count"),   # qtd de itens no pedido
        price_total     = ("price",         "sum"),     # soma dos precos
        freight_total   = ("freight_value", "sum"),     # r=+0.18: soma dos fretes
        n_sellers       = ("seller_id",     "nunique"), # qtd de vendedores distintos
        price_max       = ("price",         "max"),     # auxiliar para price_range
        price_min       = ("price",         "min"),     # auxiliar para price_range
        product_id_1st  = ("product_id",    "first"),   # produto principal do pedido
        seller_id_1st   = ("seller_id",     "first"),   # vendedor principal
    ).reset_index()

    # price_range: amplitude de preco no pedido (max - min)
    # Pedidos heterogeneos (itens de valores muito diferentes) tendem a ter
    # logistica mais complexa (ex: celular + capa de celular)
    ia["price_range"] = ia["price_max"] - ia["price_min"]

    # freight_ratio: proporcao do frete em relacao ao preco dos produtos
    # freight_total / (price_total + 0.01) -> +0.01 evita divisao por zero
    # .clip(upper=5.0) limita outliers extremos (frete maior que 5x o preco)
    # r = +0.10: ratio alto indica produto pesado/volumoso OU grande distancia
    ia["freight_ratio"] = (ia["freight_total"] / (ia["price_total"] + 0.01)).clip(upper=5.0)

    # avg_price_per_item: ticket medio por item = preco_total / n_itens
    # Proxy do tipo de produto: eletronicos (ticket alto) vs cosmeticos (baixo)
    ia["avg_price_per_item"] = ia["price_total"] / ia["n_items"]

    df = df.merge(ia.drop(columns=["price_max", "price_min"]), on="order_id", how="left")

    # ── [5] PAGAMENTOS ────────────────────────────────────────────────────────
    pay = dfs["payments"]
    pa = pay.groupby("order_id").agg(
        payment_value_total      = ("payment_value",        "sum"),  # inclui juros de parcelamento
        payment_installments_max = ("payment_installments", "max"),  # maximo de parcelas
    ).reset_index()

    # payment_type: tipo de pagamento da 1a transacao (payment_sequential=1)
    # Categorias: credit_card (11.4d), boleto (12.6d), voucher (11.3d), debit_card (10.1d)
    # Boleto demora mais: compensacao bancaria ocorre no proximo dia util (D+1 a D+3)
    pay_type = (pay.sort_values("payment_sequential")
                   .groupby("order_id")["payment_type"].first().reset_index())
    pa = pa.merge(pay_type, on="order_id")
    df = df.merge(pa, on="order_id", how="left")

    # ── [6] PRODUTOS + TRADUCAO ───────────────────────────────────────────────
    prod = dfs["products"].merge(dfs["translation"], on="product_category_name", how="left")

    # volume_cm3: comprimento x altura x largura em centimetros cubicos
    # Captura o espaco que o produto ocupa na embalagem e no veiculo de entrega
    prod["volume_cm3"] = (prod["product_length_cm"] * prod["product_height_cm"]
                          * prod["product_width_cm"])

    # densidade_g_cm3: peso / volume (gramas por cm3)
    # Por que e util? Peso e volume juntos nao capturam tudo:
    #   - Uma barra de ferro de 2kg e pequena (densidade alta ~7.8 g/cm3)
    #   - Uma almofada de 2kg e enorme (densidade baixa ~0.02 g/cm3)
    # A densidade diferencia os dois casos e tem impacto diferente no frete.
    # .replace(0, np.nan): evita divisao por zero (produtos sem dimensoes cadastradas)
    # .clip(upper=10.0): limita outliers (metais pesados ou erros de cadastro)
    prod["densidade_g_cm3"] = (
        prod["product_weight_g"] / prod["volume_cm3"].replace(0, np.nan)
    ).clip(upper=10.0)

    df = df.merge(
        prod[["product_id", "product_category_name_english",
              "product_weight_g", "volume_cm3", "densidade_g_cm3", "product_photos_qty"]],
        left_on="product_id_1st", right_on="product_id", how="left"
    )

    # ── [7] GEOLOCALIZACAO - mediana por CEP ─────────────────────────────────
    # A tabela geolocation tem MULTIPLAS entradas por CEP (diferentes fontes GPS).
    # Mediana por CEP: mais robusta que media em presenca de coordenadas erroneas.
    geo_med = (dfs["geolocation"]
               .groupby("geolocation_zip_code_prefix")[["geolocation_lat", "geolocation_lng"]]
               .median().reset_index())

    # Coordenadas do CLIENTE (via customer_zip_code_prefix -> lat/lng mediana)
    cg = (dfs["customers"][["customer_id", "customer_zip_code_prefix"]]
          .merge(geo_med, left_on="customer_zip_code_prefix",
                 right_on="geolocation_zip_code_prefix", how="left")
          .rename(columns={"geolocation_lat": "clat", "geolocation_lng": "clng"}))

    # Coordenadas do VENDEDOR PRINCIPAL (via seller_zip_code_prefix -> lat/lng mediana)
    sg = (dfs["sellers"][["seller_id", "seller_zip_code_prefix"]]
          .merge(geo_med, left_on="seller_zip_code_prefix",
                 right_on="geolocation_zip_code_prefix", how="left")
          .rename(columns={"geolocation_lat": "slat", "geolocation_lng": "slng"}))

    df = df.merge(cg[["customer_id", "clat", "clng"]], on="customer_id", how="left")
    df = df.merge(sg[["seller_id", "slat", "slng"]],
                  left_on="seller_id_1st", right_on="seller_id", how="left")

    # dist_km | r = +0.44 (feature mais correlacionada com o target)
    # Distancia geodesica em km entre cliente e vendedor principal.
    # Filtro: coordenadas devem estar dentro dos limites do Brasil (-35 a +6 graus lat)
    # para evitar coordenadas invalidas que gerariam distancias absurdas.
    valid = (df["clat"].notna() & df["slat"].notna()
             & df["clat"].between(-35, 6) & df["slat"].between(-35, 6))
    df["dist_km"] = np.nan  # inicializar como NaN; preencher apenas onde valid=True
    df.loc[valid, "dist_km"] = haversine_vec(
        df.loc[valid, "clat"].values, df.loc[valid, "clng"].values,
        df.loc[valid, "slat"].values, df.loc[valid, "slng"].values
    )

    # delta_lat | r = +0.23
    # Diferenca de latitude: cliente - vendedor
    # Positivo = cliente esta mais ao Norte que o vendedor
    # dist_km e um escalar (nao captura direcao): AM->SP e SP->AM tem dist_km identica
    # mas logisticas completamente diferentes. delta_lat/lng capturam a direcao da rota.
    df["delta_lat"] = df["clat"] - df["slat"]

    # delta_lng | r = +0.12
    # Diferenca de longitude: cliente - vendedor
    # Positivo = cliente esta mais a Leste que o vendedor
    df["delta_lng"] = df["clng"] - df["slng"]

    # Renomear lat/lng do cliente para uso direto como features
    df = df.rename(columns={"clat": "geolocation_lat", "clng": "geolocation_lng"})

    # faixa_dist_km | r = +0.47 (feature mais correlacionada)
    # Categorizacao da distancia em 6 faixas operacionais de logistica:
    # 0 = local (0-100km)          : entrega same-day ou D+1 possivel
    # 1 = regional (100-300km)     : rota intra-estado de caminhao
    # 2 = inter-regional (300-600km): entre estados proximos
    # 3 = longa (600-1000km)       : necessita hub logistico intermediario
    # 4 = muito longa (1000-2000km): 2+ etapas de transporte
    # 5 = extrema (2000km+)        : Norte/Nordeste distante, possivel aviacao
    # right=False: inclui o limite esquerdo [a, b)
    df["faixa_dist_km"] = pd.cut(
        df["dist_km"], bins=[0, 100, 300, 600, 1000, 2000, 10000],
        labels=[0, 1, 2, 3, 4, 5], right=False
    ).astype(float)

    # ── [8] UF E MACRORREGIAO ─────────────────────────────────────────────────
    df = df.merge(dfs["sellers"][["seller_id", "seller_state"]],
                  left_on="seller_id_1st", right_on="seller_id",
                  how="left", suffixes=("", "_dup"))
    df = df.merge(
        dfs["customers"][["customer_id", "customer_state", "customer_zip_code_prefix"]],
        on="customer_id", how="left"
    )

    # mesma_uf | r = -0.41 | FEATURE #1 DO MODELO (Gain XGBoost = 0.77)
    # Flag binaria: 1 se cliente e vendedor estao no mesmo estado, 0 caso contrario.
    # Intra-UF: media de 7.3 dias. Inter-UF: media de 14.1 dias. Diferenca de 93%.
    # Logica: rotas intra-estaduais sao mais curtas, mais diretas e usam
    # transportadoras regionais mais ageis do que os Correios nacionais.
    df["mesma_uf"] = (df["customer_state"] == df["seller_state"]).astype(int)

    # regiao_cliente / regiao_vendedor
    # .map(REGIAO_MAP) substitui cada UF pela sua macrorregiao correspondente
    # NaN onde a UF nao consta no dicionario (nao deve ocorrer com dados validos)
    df["regiao_cliente"]  = df["customer_state"].map(REGIAO_MAP)
    df["regiao_vendedor"] = df["seller_state"].map(REGIAO_MAP)

    # mesma_regiao | r = -0.33
    # Flag: 1 se cliente e vendedor estao na mesma macrorregiao.
    # Captura padrao intermediario entre mesma_uf (granular) e dist_km (continuo):
    # pedidos dentro do Sudeste sao mais rapidos do que entre Sudeste e Norte,
    # mesmo que a distancia seja similar em km.
    df["mesma_regiao"] = (df["regiao_cliente"] == df["regiao_vendedor"]).astype(int)

    # ── [9] HISTORICO DO VENDEDOR ─────────────────────────────────────────────
    # Calculado com todo o historico do vendedor nos dados (uso academico).
    # Em PRODUCAO: calcular apenas com pedidos ANTERIORES ao pedido atual
    # para evitar data leakage temporal.
    seller_stats = (
        df[["seller_id_1st", "dias_entrega"]]
        .groupby("seller_id_1st")
        .agg(
            # seller_avg_delivery | r = +0.35
            # Media historica de dias de entrega do vendedor.
            # Vendedores com operacao eficiente entregam consistentemente mais rapido.
            seller_avg_delivery = ("dias_entrega", "mean"),

            # seller_std_delivery | r = +0.15
            # Desvio padrao historico do vendedor.
            # Desvio alto = vendedor inconsistente (algumas entregas rapidas, outras lentas).
            # NaN para vendedores com apenas 1 pedido (std indefinido com 1 ponto).
            seller_std_delivery = ("dias_entrega", "std"),

            # seller_n_orders
            # Total de pedidos do vendedor no historico.
            # Proxy de experiencia logistica: vendedores experientes tem processos mais maduros.
            seller_n_orders     = ("dias_entrega", "count"),
        )
        .reset_index().rename(columns={"seller_id_1st": "_sid"})
    )
    df = df.merge(seller_stats, left_on="seller_id_1st", right_on="_sid", how="left")

    # Imputar NaN de seller_std_delivery com a mediana global
    # (vendedores com apenas 1 pedido nao tem std calculavel)
    df["seller_std_delivery"] = df["seller_std_delivery"].fillna(
        df["seller_std_delivery"].median()
    )

    # media_dias_uf_cliente | r = +0.46 (3a feature mais correlacionada)
    # Target encoding da UF destino: media historica de dias_entrega para cada UF.
    # Captura a infraestrutura logistica da regiao de destino:
    # SP: 8.1 dias | AM: 24.8 dias | media nacional: 11.6 dias
    uf_mean = df.groupby("customer_state")["dias_entrega"].mean().rename("media_dias_uf_cliente")
    df = df.merge(uf_mean, on="customer_state", how="left")

    # ── [10] REMOVER COLUNAS AUXILIARES ───────────────────────────────────────
    # IDs de alta cardinalidade nao devem ser usados como features (overfitting).
    # Chaves temporarias criadas durante os merges tambem sao descartadas.
    df = df.drop(columns=[
        "product_id",               # ID do produto (alta cardinalidade)
        "seller_id",                # ID do vendedor (alta cardinalidade)
        "seller_id_dup",            # duplicata criada no merge de sellers
        "_sid",                     # chave temporaria do merge de seller_stats
        "product_id_1st",           # chave temporaria para merge de produtos
        "seller_id_1st",            # chave temporaria para merge de sellers
        "customer_zip_code_prefix", # CEP substituido por lat/lng mediana
        "customer_id",              # ID do cliente (alta cardinalidade)
        "order_id",                 # ID do pedido (alta cardinalidade)
        "slat", "slng",             # lat/lng do vendedor (substituidas por dist_km/delta_lat)
    ], errors="ignore")

    return df.reset_index(drop=True)


df_raw = construir_dataset(dfs)
print(f"Dataset bruto: {df_raw.shape}")
display(df_raw.head(5))

---
## Secao 3 - Limpeza dos Dados

In [ ]:
# =============================================================================
# SECAO 3 - FILTRAGEM E LIMPEZA
# =============================================================================
# Regra geral: remover outliers APENAS no target, nao nas features.
# Outliers nas features (ex: produto de 40kg) sao valores reais e informativos.
# Outliers no target (>46 dias) sao eventos excepcionais (greves, desastres)
# que distorceriam os coeficientes sem representar o comportamento normal.
#
# Por que P99=46d e nao P95=38d?
# P95 descartaria entregas legitimas do Norte (AM, RR, AP) que regularmente
# levam 20-30 dias. P99 remove apenas o 1% mais extremo (~880 pedidos de 96k).

def limpar_dataset(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    df = df.dropna(subset=["dias_entrega"])           # 8 linhas: data_entrega nao registrada
    df = df[df["dias_entrega"] > 0]                   # impossivel fisicamente: timestamp corrompido
    df = df[df["dias_entrega"] <= OUTLIER_P99]        # acima do P99: evento excepcional (~0.9%)
    df = df.dropna(subset=["estimativa_prazo"])
    df = df[df["estimativa_prazo"] > 0]               # estimativa invalida: dado corrompido
    logger.info(f"Limpeza: {n:,} -> {len(df):,} linhas ({n-len(df):,} removidas)")
    return df.reset_index(drop=True)

df_clean = limpar_dataset(df_raw)

# Estatisticas descritivas do target
# skewness = 1.43: assimetria positiva (cauda direita = pedidos lentos do Norte)
# media (11.6d) > mediana (10d): confirmacao da assimetria
target_stats = df_clean["dias_entrega"].describe().round(2).to_frame().T
target_stats.insert(0, "skewness", round(df_clean["dias_entrega"].skew(), 3))
target_stats.insert(0, "n_pedidos", len(df_clean))
display(target_stats)

# Inventario de nulos com estrategia de imputacao
nulos = df_clean.isnull().sum()
nulos = nulos[nulos > 0].reset_index()
nulos.columns = ["feature", "n_nulos"]
nulos["pct_%"] = (nulos["n_nulos"] / len(df_clean) * 100).round(3)
nulos["tratamento"] = nulos["feature"].apply(lambda f:
    "SimpleImputer(most_frequent) + OrdinalEncoder"  # moda para categoricas
    if f in {"product_category_name_english", "regiao_cliente", "regiao_vendedor"}
    else "KNNImputer(n_neighbors=5) + StandardScaler"  # KNN para numericas
)
display(nulos)

df_clean.to_csv(OUT / "olist_dataset_enriquecido.csv", index=False)
logger.info(f"Dataset exportado: {df_clean.shape}")

---
## Secao 4 - EDA (9 analises)

In [ ]:
# 4.1 DISTRIBUICAO DO TARGET
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_clean["dias_entrega"], bins=46, color=PALETA, edgecolor="white", alpha=0.9)
axes[0].axvline(df_clean["dias_entrega"].mean(), color="red", ls="--", lw=1.5,
                label=f"Media {df_clean['dias_entrega'].mean():.1f}d")
axes[0].axvline(df_clean["dias_entrega"].median(), color="orange", ls="--", lw=1.5,
                label=f"Mediana {df_clean['dias_entrega'].median():.0f}d")
axes[0].set_title("Distribuicao de dias_entrega (target)")
axes[0].set_xlabel("Dias"); axes[0].set_ylabel("Frequencia")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[0].legend()

meses = sorted(df_clean["mes_compra"].unique())
axes[1].boxplot([df_clean[df_clean["mes_compra"]==m]["dias_entrega"].values for m in meses],
                positions=meses, widths=0.6, patch_artist=True, showfliers=False,
                boxprops=dict(facecolor=PALETA, alpha=0.7), medianprops=dict(color="white", lw=2))
axes[1].set_title("Sazonalidade: dias_entrega por Mes")
axes[1].set_xlabel("Mes"); axes[1].set_ylabel("Dias"); axes[1].set_xticks(meses)
axes[1].set_xticklabels(["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"], fontsize=8)
plt.tight_layout(); plt.savefig(OUT/"fig1_target.png", bbox_inches="tight"); plt.show()
display(df_clean.groupby("mes_compra")["dias_entrega"].agg(media="mean",mediana="median",std="std",n="count").round(2).reset_index())

In [ ]:
# 4.2 CORRELACOES - Pearson r mede correlacao LINEAR entre variaveis continuas
# Varia de -1 (negativa perfeita) a +1 (positiva perfeita). r=0: sem correlacao linear.
# Interpretacao: |r|>0.35 = forte; 0.10-0.35 = moderada; <0.10 = fraca
num_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
corr_all = (df_clean[num_cols].corr()["dias_entrega"].drop("dias_entrega")
            .sort_values(key=abs, ascending=False).head(20))

fig, ax = plt.subplots(figsize=(10, 8))
cores_bar = ["#2563EB" if v > 0 else "#DC2626" for v in corr_all.values]
bars = ax.barh(corr_all.index[::-1], corr_all.values[::-1], color=cores_bar[::-1], edgecolor="white", alpha=0.85)
ax.axvline(0, color="gray", lw=0.8)
for bar, val in zip(bars, corr_all.values[::-1]):
    ax.text(val+(0.005 if val>=0 else -0.005), bar.get_y()+bar.get_height()/2,
            f"{val:+.3f}", va="center", ha="left" if val>=0 else "right", fontsize=9)
ax.set_title("Correlacao de Pearson - Features x dias_entrega (top 20)")
ax.set_xlabel("r de Pearson"); ax.set_xlim(-0.55, 0.62)
plt.tight_layout(); plt.savefig(OUT/"fig2_correlacoes.png", bbox_inches="tight"); plt.show()
display(corr_all.reset_index().rename(columns={"index":"feature","dias_entrega":"pearson_r"})
        .assign(pearson_r=lambda d: d["pearson_r"].round(4))
        .assign(forca=lambda d: d["pearson_r"].apply(
            lambda r: "forte +" if r>0.35 else ("forte -" if r<-0.35 else ("moderada" if abs(r)>0.10 else "fraca")))))

In [ ]:
# 4.3 MATRIZ DE CORRELACAO - detectar multicolinearidade entre features
# Multicolinearidade: duas features muito correlacionadas entre SI (nao com o target).
# Par mais colinear: faixa_dist_km x dist_km (r~0.97) - esperado, faixa e derivada de dist_km.
# Para arvores (XGBoost, RF): impacto menor. Para Ridge: pode instabilizar coeficientes.
key_feats = ["dias_entrega","dist_km","media_dias_uf_cliente","estimativa_prazo",
             "mesma_uf","seller_avg_delivery","mesma_regiao","geolocation_lat",
             "freight_total","delta_lat","seller_std_delivery",
             "dias_ate_aprova_h","freight_ratio","product_weight_g","volume_cm3"]
corr_mat = df_clean[key_feats].corr()
mask = np.triu(np.ones_like(corr_mat, dtype=bool))  # mascara triangulo superior (evita duplicacao)
fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.4, annot_kws={"size":8}, ax=ax)
ax.set_title("Matriz de Correlacao - 15 Features Mais Relevantes", pad=15)
plt.tight_layout(); plt.savefig(OUT/"fig3_heatmap.png", bbox_inches="tight"); plt.show()

In [ ]:
# 4.4 OUTLIERS (criterio IQR de Tukey)
# IQR = Q3 - Q1 (amplitude do intervalo central de 50% dos dados)
# Outlier: valor fora de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
# Outliers NAO sao removidos das features: sao valores reais e informativos.
features_box = ["price_total","freight_total","product_weight_g","volume_cm3",
                "payment_value_total","dias_ate_aprova_h","n_items","dist_km","seller_avg_delivery"]
fig, axes = plt.subplots(3, 3, figsize=(15, 10)); axes = axes.flatten()
outlier_report = []
for i, col in enumerate(features_box):
    dados = df_clean[col].dropna()
    q1, q3 = dados.quantile(0.25), dados.quantile(0.75); iqr = q3 - q1
    n_out = ((dados < q1-1.5*iqr) | (dados > q3+1.5*iqr)).sum()
    axes[i].boxplot(dados, vert=True, showfliers=True,
                    flierprops=dict(marker=".", markersize=2, alpha=0.3, color="red"))
    axes[i].set_title(f"{col}\n({n_out:,} outliers IQR)", fontsize=9)
    outlier_report.append({"feature":col,"Q1":round(q1,2),"Q3":round(q3,2),"IQR":round(iqr,2),
                           "n_outliers":int(n_out),"pct_%":round(n_out/len(dados)*100,2)})
plt.suptitle("Deteccao de Outliers - Boxplot (criterio IQR)", fontsize=13)
plt.tight_layout(); plt.savefig(OUT/"fig4_outliers.png", bbox_inches="tight"); plt.show()
display(pd.DataFrame(outlier_report))

In [ ]:
# 4.5 MAPA GEOGRAFICO
estado_media = df_clean.groupby("customer_state")["dias_entrega"].mean().sort_values()
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
cores_uf = ["#DC2626" if e==estado_media.idxmax() else ("#16A34A" if e==estado_media.idxmin() else PALETA) for e in estado_media.index]
axes[0].barh(estado_media.index, estado_media.values, color=cores_uf, edgecolor="white")
axes[0].axvline(df_clean["dias_entrega"].mean(), color="gray", ls="--", lw=1, label="Media geral")
for i,(uf,val) in enumerate(zip(estado_media.index, estado_media.values)):
    axes[0].text(val+0.1, i, f"{val:.1f}d", va="center", fontsize=7)
axes[0].set_title("Tempo Medio de Entrega por UF"); axes[0].set_xlabel("Dias"); axes[0].legend()
df_geo = df_clean.dropna(subset=["geolocation_lat","geolocation_lng"])
# alpha=0.15: transparencia alta pois ha 95k pontos sobrepostos (density map)
# cmap="RdYlGn_r": vermelho=lento, verde=rapido (invertido _r)
sc = axes[1].scatter(df_geo["geolocation_lng"], df_geo["geolocation_lat"],
                     c=df_geo["dias_entrega"], cmap="RdYlGn_r", alpha=0.15, s=2, vmin=0, vmax=40)
plt.colorbar(sc, ax=axes[1], label="Dias de Entrega")
axes[1].set_xlim(-75,-34); axes[1].set_ylim(-34,6)
axes[1].set_title("Mapa: Dias de Entrega x Localizacao do Cliente")
axes[1].set_xlabel("Longitude"); axes[1].set_ylabel("Latitude")
plt.tight_layout(); plt.savefig(OUT/"fig5_geo.png", bbox_inches="tight"); plt.show()
uf_rank = (df_clean.groupby("customer_state")["dias_entrega"]
           .agg(media="mean",mediana="median",std="std",n_pedidos="count").round(2)
           .sort_values("media", ascending=False).reset_index().rename(columns={"customer_state":"UF"}))
uf_rank["regiao"] = uf_rank["UF"].map(REGIAO_MAP)
display(uf_rank)

In [ ]:
# 4.6 DISTANCIA HAVERSINE x PRAZO
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
faixa_labels = ["0-100km\n(local)","100-300km\n(regional)","300-600km\n(inter-reg)",
                "600-1000km\n(longa)","1000-2000km\n(muito longa)","2000km+\n(extrema)"]
faixa_stats = df_clean.groupby("faixa_dist_km")["dias_entrega"].agg(["mean","count"]).reset_index()
cores_f = ["#16A34A","#4ADE80","#FBBF24","#F97316","#EF4444","#991B1B"]
bars = axes[0].bar(range(len(faixa_labels)), faixa_stats["mean"].values, color=cores_f, edgecolor="white", alpha=0.9)
axes[0].set_xticks(range(len(faixa_labels))); axes[0].set_xticklabels(faixa_labels, fontsize=8)
axes[0].set_title("Prazo Medio por Faixa de Distancia (Haversine)"); axes[0].set_ylabel("Dias")
axes[0].axhline(df_clean["dias_entrega"].mean(), color="gray", ls="--", lw=1)
for bar, val in zip(bars, faixa_stats["mean"].values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2, f"{val:.1f}d", ha="center", fontsize=9, fontweight="bold")
# np.polyfit: ajusta polinomio grau 1 (reta) -> coef_angular e intercepto
df_s = df_clean.dropna(subset=["dist_km"]).sample(10_000, random_state=42)
axes[1].scatter(df_s["dist_km"], df_s["dias_entrega"], alpha=0.08, s=4, color=PALETA)
z = np.polyfit(df_s["dist_km"], df_s["dias_entrega"], 1)
xl = np.linspace(0, 3800, 200)
axes[1].plot(xl, np.poly1d(z)(xl), color="red", lw=2, label="Tendencia (r=+0.44)")
axes[1].set_xlabel("dist_km"); axes[1].set_ylabel("dias_entrega")
axes[1].set_title("Dispersao: dist_km x dias_entrega (amostra 10k)"); axes[1].legend()
plt.tight_layout(); plt.savefig(OUT/"fig6_distancia.png", bbox_inches="tight"); plt.show()
display(faixa_stats.assign(faixa=faixa_labels).rename(columns={"mean":"media_dias","count":"n_pedidos"})[["faixa","media_dias","n_pedidos"]].round(2))

In [ ]:
# 4.7-4.9 EDA COMPLEMENTAR (vendedor, financeiro, produto)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# Violin plot: mostra distribuicao completa com KDE (Kernel Density Estimation)
# mais informativo que boxplot para distribuicoes multimodais
for idx, v in enumerate([0, 1]):
    dados = df_clean[df_clean["mesma_uf"]==v]["dias_entrega"]
    vp = axes[0].violinplot(dados, positions=[idx], showmedians=True, showextrema=False, widths=0.6)
    for pc in vp["bodies"]: pc.set_facecolor("#EF4444" if v==0 else "#16A34A"); pc.set_alpha(0.7)
axes[0].set_xticks([0,1]); axes[0].set_xticklabels(["UF diferente\n(14,1d)","Mesma UF\n(7,3d)"])
axes[0].set_title("Violin: mesma_uf x dias_entrega"); axes[0].set_ylabel("Dias")
regioes = df_clean.groupby("regiao_cliente")["dias_entrega"].median().sort_values(ascending=False).index.tolist()
bp = axes[1].boxplot([df_clean[df_clean["regiao_cliente"]==r]["dias_entrega"].values for r in regioes],
                     labels=regioes, patch_artist=True, showfliers=False, medianprops=dict(color="white",lw=2))
for patch, cor in zip(bp["boxes"],["#DC2626","#EF4444","#FBBF24","#16A34A","#2563EB"]): patch.set_facecolor(cor); patch.set_alpha(0.7)
axes[1].set_title("Boxplot: Macrorregiao x dias_entrega"); axes[1].set_ylabel("Dias")
plt.setp(axes[1].xaxis.get_majorticklabels(), fontsize=8)
df_sell = df_clean.dropna(subset=["seller_avg_delivery"]).sample(8_000, random_state=42)
axes[2].scatter(df_sell["seller_avg_delivery"].clip(upper=40), df_sell["dias_entrega"], alpha=0.08, s=4, color="#7C3AED")
z2 = np.polyfit(df_sell["seller_avg_delivery"].clip(upper=40), df_sell["dias_entrega"], 1)
axes[2].plot(np.linspace(0,40,100), np.poly1d(z2)(np.linspace(0,40,100)), color="red", lw=2, label="r = +0.35")
axes[2].set_xlabel("seller_avg_delivery (dias)"); axes[2].set_ylabel("dias_entrega")
axes[2].set_title("Historico Vendedor x Prazo Real"); axes[2].legend()
plt.tight_layout(); plt.savefig(OUT/"fig7_geo_seller.png", bbox_inches="tight"); plt.show()
display(df_clean.groupby("regiao_cliente")["dias_entrega"].agg(media="mean",mediana="median",std="std",n="count").round(2).sort_values("media",ascending=False).reset_index())

---
## Secao 5 - Pre-processamento (ColumnTransformer)

In [ ]:
# =============================================================================
# SECAO 5 - PRE-PROCESSAMENTO
# =============================================================================
TARGET = "dias_entrega"

# NUMERICAS (32): recebem KNNImputer(k=5) -> StandardScaler
# CATEGORICAS (6): recebem SimpleImputer(mode) -> OrdinalEncoder
NUM_FEATURES: List[str] = [
    # Temporais (7)
    "estimativa_prazo","dias_ate_aprova_h","dia_semana_compra","hora_compra","mes_compra","fim_de_semana","periodo_dia",
    # Financeiras (7)
    "price_total","freight_total","payment_value_total","payment_installments_max","freight_ratio","avg_price_per_item","price_range",
    # Itens (2)
    "n_items","n_sellers",
    # Produto (4)
    "product_weight_g","volume_cm3","densidade_g_cm3","product_photos_qty",
    # Geograficas (9)
    "geolocation_lat","geolocation_lng","dist_km","delta_lat","delta_lng","faixa_dist_km","mesma_uf","mesma_regiao","media_dias_uf_cliente",
    # Seller (3)
    "seller_avg_delivery","seller_std_delivery","seller_n_orders",
]
CAT_FEATURES: List[str] = [
    "payment_type","customer_state","seller_state","regiao_cliente","regiao_vendedor","product_category_name_english",
]
ALL_FEATURES = NUM_FEATURES + CAT_FEATURES  # 32 + 6 = 38 features

if not set(ALL_FEATURES).issubset(set(df_clean.columns)):
    logger.warning("Recarregando CSV enriquecido...")
    df_clean = pd.read_csv(OUT / "olist_dataset_enriquecido.csv")

X = df_clean[ALL_FEATURES].copy()
y = df_clean[TARGET].copy()

# Split 80/20: random_state=42 garante o MESMO split toda vez (reprodutibilidade)
# Os 19.115 pedidos de teste NUNCA serao vistos durante treino ou tuning.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE)
print(f"Treino: {X_train.shape[0]:,} | Teste: {X_test.shape[0]:,} | Features: {len(ALL_FEATURES)}")

# Analise de nulos NO TREINO (nao no dataset completo: o teste nao deve influenciar nada)
nulos_treino = X_train.isnull().sum(); nulos_treino = nulos_treino[nulos_treino > 0].reset_index()
nulos_treino.columns = ["feature","n_nulos"]
nulos_treino["pct_%"] = (nulos_treino["n_nulos"] / len(X_train) * 100).round(3)
nulos_treino["tratamento"] = nulos_treino["feature"].apply(lambda f:
    "SimpleImputer" if f in {"product_category_name_english","regiao_cliente","regiao_vendedor"} else "KNNImputer(k=5)")
display(nulos_treino)

# Pipeline NUMERICO:
# KNNImputer(n_neighbors=5): imputa pelo valor medio dos 5 vizinhos mais proximos
#   no espaco de todas as features numericas. Superior ao SimpleImputer(median)
#   porque usa as correlacoes entre features (ex: produto pesado com volume nulo
#   recebe o volume de produtos com peso similar, nao a mediana global).
# StandardScaler: z-score -> (x - media) / std, resultando em media=0, std=1.
#   Necessario para Ridge (reg. L2 e sensivel a escala dos coeficientes).
numeric_pipe = Pipeline([("imputer", KNNImputer(n_neighbors=5)), ("scaler", StandardScaler())])

# Pipeline CATEGORICO:
# SimpleImputer(most_frequent): preenche NaN com a categoria mais comum (moda).
# OrdinalEncoder: mapeia cada categoria unica para um inteiro (0, 1, 2, ...).
#   Por que nao OneHotEncoder? Com 71 categorias em product_category_name_english,
#   OHE geraria 71+ colunas binarias (esparsas). OrdinalEncoder mantem 1 coluna.
#   handle_unknown="use_encoded_value": categorias novas no teste recebem -1 (nao gera erro).
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

# ColumnTransformer: aplica os dois pipelines em paralelo, cada um no seu grupo.
# FUNDAMENTAL: fit() so no treino -> parametros (media, std, categorias) estimados
# APENAS com dados de treino. Previne data leakage ao transformar o conjunto de teste.
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipe, NUM_FEATURES),
    ("cat", categorical_pipe, CAT_FEATURES),
], remainder="drop")
print("Preprocessor configurado")

---
## Secao 6 - Benchmark de Modelos

In [ ]:
# =============================================================================
# SECAO 6 - BENCHMARK DE 5 MODELOS
# =============================================================================
def avaliar_regressao(nome: str, y_true, y_pred: np.ndarray) -> Dict:
    """
    Metricas de regressao:
    RMSE = sqrt(MSE): penaliza erros grandes quadraticamente. Mesma unidade do target (dias).
    MAE = media(|real - predito|): robusto a outliers. Interpretacao direta: "erra X dias em media".
    MAPE = media(|real - predito| / |real|) * 100: erro percentual para comunicar a stakeholders.
    R2 = 1 - SS_res/SS_tot: proporcao da variancia explicada. R2=1: perfeito. R2=0: igual a naive (media).
    """
    return {
        "modelo": nome,
        "RMSE":   round(float(np.sqrt(mean_squared_error(y_true, y_pred))), 4),
        "MAE":    round(float(mean_absolute_error(y_true, y_pred)), 4),
        "MAPE_%": round(float(np.mean(np.abs((np.array(y_true)-y_pred)/np.clip(y_true,1,None)))*100), 4),
        "R2":     round(float(r2_score(y_true, y_pred)), 4),
    }

MODELOS_PIPELINE: Dict[str, Pipeline] = {
    # Ridge: regressao linear + regularizacao L2 (alpha*||w||^2 penaliza coeficientes grandes)
    # alpha=10.0: forca de regularizacao. Baseline linear - se modelos complexos nao forem
    # muito melhores, a relacao e essencialmente linear.
    "Ridge": Pipeline([("pre",preprocessor),("reg",Ridge(alpha=10.0))]),

    # DecisionTree: arvore simples. max_depth=8 limita overfitting.
    # min_samples_leaf=50: no terminal precisa de pelo menos 50 amostras.
    # Baseline nao-linear, interpretavel por regras mas alta variancia.
    "DecisionTree": Pipeline([("pre",preprocessor),("reg",DecisionTreeRegressor(max_depth=8,min_samples_leaf=50,random_state=RANDOM_STATE))]),

    # RandomForest: BAGGING de 150 arvores. Cada arvore ve subset aleatorio de dados E features.
    # Predicao = MEDIA de todas as arvores. Resultado: menor variancia que DecisionTree.
    "RandomForest": Pipeline([("pre",preprocessor),("reg",RandomForestRegressor(n_estimators=150,max_depth=12,min_samples_leaf=20,random_state=RANDOM_STATE,n_jobs=-1))]),

    # GradientBoosting: BOOSTING SEQUENCIAL. Cada arvore corrige os RESIDUOS da anterior.
    # learning_rate=0.08: shrinkage (cada arvore contribui apenas 8% do valor).
    # subsample=0.8: usa 80% dos dados por arvore (stochastic boosting, anti-overfit).
    "GradientBoosting": Pipeline([("pre",preprocessor),("reg",GradientBoostingRegressor(n_estimators=150,learning_rate=0.08,max_depth=5,subsample=0.8,random_state=RANDOM_STATE))]),

    # XGBoost: boosting otimizado com regularizacao L1+L2 nativa.
    # colsample_bytree=0.8: 80% das features por arvore (column bagging, anti-overfit).
    # Vantagens vs GradientBoosting: regularizacao mais granular, missing values nativos,
    # paralelizacao eficiente. Melhor RMSE e R2 no benchmark.
    "XGBoost": Pipeline([("pre",preprocessor),("reg",XGBRegressor(n_estimators=200,learning_rate=0.08,max_depth=6,subsample=0.8,colsample_bytree=0.8,random_state=RANDOM_STATE,n_jobs=-1,verbosity=0))]),
}

resultados = []
for nome, pipe in MODELOS_PIPELINE.items():
    pipe.fit(X_train, y_train)
    met = avaliar_regressao(nome, y_test, pipe.predict(X_test))
    resultados.append(met); logger.info(f"{nome:20s} RMSE={met['RMSE']:.4f} R2={met['R2']:.4f}")

resultado_df = pd.DataFrame(resultados).set_index("modelo").sort_values("RMSE")
resultado_df.to_csv(OUT/"benchmark_resultados.csv")
display(resultado_df.reset_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ml = resultado_df.index.tolist(); rmses = resultado_df["RMSE"].values; r2s = resultado_df["R2"].values
cores = ["#16A34A" if i==0 else PALETA for i in range(len(ml))]
axes[0].barh(ml[::-1], rmses[::-1], color=cores[::-1], edgecolor="white", alpha=0.9)
for i,v in enumerate(rmses[::-1]): axes[0].text(v+0.02,i,f"{v:.3f}",va="center",fontsize=9)
axes[0].set_title("RMSE - Teste (menor = melhor)"); axes[0].set_xlabel("RMSE (dias)")
axes[1].barh(ml[::-1], r2s[::-1], color=cores[::-1], edgecolor="white", alpha=0.9)
for i,v in enumerate(r2s[::-1]): axes[1].text(v+0.002,i,f"{v:.4f}",va="center",fontsize=9)
axes[1].set_title("R2 - Teste (maior = melhor)"); axes[1].set_xlabel("R2"); axes[1].set_xlim(0,1.05)
plt.suptitle("Comparacao de Modelos de Regressao - Olist",fontsize=13)
plt.tight_layout(); plt.savefig(OUT/"fig10_benchmark.png",bbox_inches="tight"); plt.show()

---
## Secao 7 - RandomizedSearchCV

In [ ]:
# =============================================================================
# SECAO 7 - AJUSTE DE HIPERPARAMETROS (RandomizedSearchCV)
# =============================================================================
# RandomizedSearch vs GridSearch:
# Grid: testa TODAS as combinacoes. Com 8 params x ~4 valores = ~65k combinacoes -> inviavel.
# Random: amostra n_iter=20 combinacoes ALEATORIAS.
# n_iter=20 x cv=5 = 100 fits totais. Bergstra & Bengio (2012): amostragem aleatoria
# e tao eficaz quanto grid para funcoes de perda suaves, a ~100x menos custo.
xgb_tunavel = Pipeline([("pre",preprocessor),("reg",XGBRegressor(random_state=RANDOM_STATE,n_jobs=-1,verbosity=0))])
param_dist = {
    "reg__n_estimators":     [150, 200, 300, 400],      # +arvores: -variancia, +custo
    "reg__learning_rate":    [0.03, 0.05, 0.08, 0.1, 0.15],  # shrinkage: menor=mais robusto
    "reg__max_depth":        [4, 5, 6, 7],              # profundidade: maior=overfit
    "reg__subsample":        [0.7, 0.8, 0.9, 1.0],     # fracao de linhas por arvore
    "reg__colsample_bytree": [0.6, 0.7, 0.8, 0.9],     # fracao de features por arvore
    "reg__min_child_weight": [1, 3, 5, 10],             # minimo de amostras no no terminal
    "reg__reg_alpha":        [0, 0.01, 0.1, 1.0],      # regularizacao L1 (esparsidade)
    "reg__reg_lambda":       [0.5, 1.0, 2.0, 5.0],     # regularizacao L2 (suaviza pesos)
}
# KFold DENTRO do treino: o conjunto de teste NUNCA e visto durante o tuning
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
# scoring="neg_root_mean_squared_error": sklearn maximiza -> negativo do RMSE menor = melhor
# refit=True: re-treina com melhores params no dataset inteiro de treino ao final
random_search = RandomizedSearchCV(xgb_tunavel, param_dist, n_iter=20, cv=kf,
                                   scoring="neg_root_mean_squared_error",
                                   random_state=RANDOM_STATE, n_jobs=-1, verbose=1, refit=True)
random_search.fit(X_train, y_train)
best_params = pd.DataFrame([{"hiperparametro": p.replace("reg__",""), "valor": v}
                            for p, v in random_search.best_params_.items()])
display(best_params)
logger.info(f"RMSE CV (melhor): {-random_search.best_score_:.4f} dias")
met_tunado = avaliar_regressao("XGBoost (tunado)", y_test, random_search.predict(X_test))
display(pd.DataFrame([met_tunado]))

---
## Secao 8 - Residuos e Feature Importance

In [ ]:
# =============================================================================
# SECAO 8 - ANALISE DE RESIDUOS
# =============================================================================
# Residuo = y_real - y_predito
# Um bom modelo deve ter residuos:
# 1. Centrados em zero (sem vies sistematico -> Bias ~= 0)
# 2. Sem padrao em funcao de y_predito (homocedasticidade)
# 3. Distribuidos de forma aproximadamente normal
y_pred_best = MODELOS_PIPELINE["XGBoost"].predict(X_test)
residuos = y_test.values - y_pred_best  # vetor de residuos

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
# Painel 1: Real vs Predito - pontos proximos da diagonal y=x indicam boas predicoes
axes[0].scatter(y_pred_best, y_test.values, alpha=0.1, s=4, color=PALETA)
axes[0].plot([0,46],[0,46],"r--",lw=1.5,label="Perfeito")
axes[0].set_xlabel("Predito (dias)"); axes[0].set_ylabel("Real (dias)")
axes[0].set_title("Real vs. Predito - XGBoost"); axes[0].legend()
# Painel 2: Residuos vs Predito - padrao em funil indica heterocedasticidade (problema)
axes[1].scatter(y_pred_best, residuos, alpha=0.1, s=4, color=PALETA)
axes[1].axhline(0, color="red", ls="--", lw=1.5)
axes[1].set_xlabel("Predito (dias)"); axes[1].set_ylabel("Residuo (dias)")
axes[1].set_title("Residuos vs. Predito")
# Painel 3: Distribuicao dos residuos - verificar normalidade e vies
# Bias = 0.014: quase zero -> modelo nao superestima nem subestima sistematicamente
axes[2].hist(residuos, bins=50, color=PALETA, edgecolor="white", alpha=0.9)
axes[2].axvline(0, color="red", ls="--", lw=1.5)
axes[2].set_xlabel("Residuo (dias)"); axes[2].set_ylabel("Frequencia")
axes[2].set_title(f"Distribuicao de Residuos | Bias = {residuos.mean():.3f}d")
plt.tight_layout(); plt.savefig(OUT/"fig11_residuos.png", bbox_inches="tight"); plt.show()
display(pd.DataFrame({
    "Metrica": ["Bias (media dos residuos)","Std dos residuos","% residuos < +-3d","% residuos < +-7d"],
    "Valor": [f"{residuos.mean():.3f}d",f"{residuos.std():.3f}d",
              f"{(np.abs(residuos)<3).mean()*100:.1f}%",f"{(np.abs(residuos)<7).mean()*100:.1f}%"]
}))

# =============================================================================
# FEATURE IMPORTANCE (Gain medio do XGBoost)
# =============================================================================
# feature_importances_ usa o "Gain" medio por feature:
# Gain = reducao media da funcao de perda (MSE) quando a feature e usada em splits.
# mesma_uf: Gain ~= 0.77 -> sozinha responde por 77% da melhoria total do modelo.
fi = MODELOS_PIPELINE["XGBoost"].named_steps["reg"].feature_importances_
fi_df = pd.DataFrame({"feature":ALL_FEATURES,"importance":fi}).sort_values("importance",ascending=False)
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(fi_df.head(20)["feature"][::-1], fi_df.head(20)["importance"][::-1], color=PALETA, edgecolor="white", alpha=0.85)
ax.set_title("Feature Importance - XGBoost (Top 20 por Gain medio)"); ax.set_xlabel("Importance Score")
plt.tight_layout(); plt.savefig(OUT/"fig12_feature_importance.png", bbox_inches="tight"); plt.show()
display(fi_df.head(15).reset_index(drop=True).round(4))

In [ ]:
# =============================================================================
# SECAO 9 - RESUMO EXECUTIVO
# =============================================================================
dataset_kpis = pd.DataFrame([
    {"KPI":"Pedidos entregues","Valor":f"{len(df_clean):,}"},
    {"KPI":"Total de features","Valor":len(ALL_FEATURES)},
    {"KPI":"Target medio","Valor":f"{df_clean['dias_entrega'].mean():.1f} dias"},
    {"KPI":"Target mediana","Valor":f"{df_clean['dias_entrega'].median():.0f} dias"},
    {"KPI":"Estado mais lento","Valor":"AM - 24,8 dias"},
    {"KPI":"Estado mais rapido","Valor":"SP - 8,1 dias"},
    {"KPI":"Modelo vencedor","Valor":"XGBoost (tunado)"},
    {"KPI":"RMSE (teste)","Valor":"5,62 dias"},
    {"KPI":"MAE (teste)","Valor":"3,94 dias"},
    {"KPI":"R2 (teste)","Valor":"0,4806"},
    {"KPI":"Bias dos residuos","Valor":"0,014 dias"},
    {"KPI":"Variancia explicada","Valor":"48,1%"},
])
display(dataset_kpis)
bench_final = resultado_df.reset_index().copy()
bench_final["vencedor"] = bench_final["modelo"].apply(lambda m: "* modelo final" if "XGBoost" in m else "")
display(bench_final.round(4))
logger.info(f"Pipeline completo. Outputs salvos em {OUT}")